# 00 — Teste dos serviços do laboratório

Este notebook confere se todos os serviços do ambiente estão funcionando:

| Serviço | O que é testado |
|---|---|
| **PostgreSQL** | conexão, encoding, replicação lógica, tabelas, publicação e usuário de CDC |
| **MinIO** | buckets e gravação/leitura de um arquivo Parquet |
| **Kafka** | tópicos, envio e leitura de uma mensagem |
| **Kafka Connect** | API REST e conector Debezium para Postgres |
| **Iceberg** | criação de tabela, gravação e leitura com PyIceberg |

**Como usar:** com o ambiente no ar (`docker compose up -d --wait`), rode todas as células em ordem (*Run → Run All Cells*).
Tudo que o teste cria (arquivos, tópicos, tabelas) é apagado no final de cada célula.

> Este notebook roda dentro do container `lab-python`. Por isso os serviços são acessados pelo nome na rede do Docker (`postgres`, `minio`, `kafka`, `connect`) e não por `localhost`.

## 1. Configuração

Lê o arquivo `.env` da raiz do projeto. As senhas são usadas para conectar, mas **nunca são exibidas**.

In [2]:
import json
import os
import time
import urllib.request
import uuid
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# O notebook roda a partir da pasta notebooks/; o .env fica na raiz do projeto
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ENV_FILE = PROJECT_ROOT / ".env"
if not ENV_FILE.is_file():
    raise FileNotFoundError(f"Arquivo .env não encontrado em {ENV_FILE}. Copie o .env.example para .env.")
load_dotenv(ENV_FILE, override=False)  # variáveis do container têm prioridade sobre o .env


def env(nome):
    """Lê uma variável obrigatória do ambiente; falha com mensagem clara se faltar."""
    valor = os.environ.get(nome, "").strip()
    if not valor:
        raise ValueError(f"Variável {nome} não definida no .env")
    return valor


# Resultado de cada teste: serviço, status e detalhe
resultados = []


def registrar(servico, ok, detalhe):
    resultados.append({"serviço": servico, "status": "OK" if ok else "FALHOU", "detalhe": detalhe})
    print(f"[{'OK' if ok else 'FALHOU'}] {servico}: {detalhe}")


def limpar(descricao, acao):
    """Apaga o que o teste criou. Se o serviço estiver fora do ar, avisa sem esconder a falha principal."""
    try:
        acao()
    except Exception as erro:
        print(f"[AVISO] limpeza de {descricao} não concluída: {type(erro).__name__}: {erro}")


print("Projeto:", PROJECT_ROOT)
print("Postgres:", env("POSTGRES_HOST"), "| MinIO:", env("S3_ENDPOINT_URL"),
      "| Kafka:", env("KAFKA_BOOTSTRAP_SERVERS"), "| Connect:", env("KAFKA_CONNECT_URL"))

Projeto: /app
Postgres: postgres | MinIO: http://minio:9000 | Kafka: kafka:29092 | Connect: http://connect:8083


## 2. PostgreSQL

Conecta com o usuário administrador (`lab_admin`) e confere a configuração necessária para os cenários:
- `wal_level = logical`, necessário para o CDC (cenários 2 e 3);
- as 7 tabelas e a quantidade de linhas de cada uma;
- a publicação usada pelo CDC e o usuário de replicação `cdc_reader`.

In [3]:
import psycopg2


def conectar_postgres(usuario, senha):
    return psycopg2.connect(
        host=env("POSTGRES_HOST"), port=int(env("POSTGRES_PORT")), dbname=env("POSTGRES_DB"),
        user=usuario, password=senha, client_encoding="UTF8", application_name="notebook_teste",
    )


try:
    conn = conectar_postgres(env("POSTGRES_USER"), env("POSTGRES_PASSWORD"))
    with conn, conn.cursor() as cur:
        cur.execute("SELECT split_part(version(), ' ', 2), current_setting('server_encoding'), "
                    "current_setting('wal_level')")
        versao, encoding, wal_level = cur.fetchone()

        cur.execute("SELECT table_name FROM information_schema.tables "
                    "WHERE table_schema = 'public' AND table_type = 'BASE TABLE' ORDER BY table_name")
        tabelas = [linha[0] for linha in cur.fetchall()]
        contagens = {}
        for tabela in tabelas:
            cur.execute(f'SELECT count(*) FROM "{tabela}"')
            contagens[tabela] = cur.fetchone()[0]

        cur.execute("SELECT count(*) FROM pg_publication_tables WHERE pubname = %s", (env("CDC_PUBLICATION"),))
        tabelas_publicadas = cur.fetchone()[0]
    conn.close()

    conn_cdc = conectar_postgres(env("CDC_USER"), env("CDC_PASSWORD"))
    with conn_cdc, conn_cdc.cursor() as cur:
        cur.execute("SELECT rolreplication, rolsuper FROM pg_roles WHERE rolname = current_user")
        cdc_replicacao, cdc_superuser = cur.fetchone()
    conn_cdc.close()

    display(pd.DataFrame(contagens.items(), columns=["tabela", "linhas"]))
    ok = (wal_level == "logical" and len(tabelas) == 7 and tabelas_publicadas == 7
          and cdc_replicacao and not cdc_superuser)
    registrar("PostgreSQL", ok,
              f"versão {versao}, {encoding}, wal_level={wal_level}, {len(tabelas)} tabelas, "
              f"{tabelas_publicadas} na publicação, cdc_reader com replicação={cdc_replicacao} "
              f"e superuser={cdc_superuser}")
    if sum(contagens.values()) == 0:
        print("Dica: a base está vazia. Para popular: docker compose exec python python src/populate.py seed")
except Exception as erro:
    registrar("PostgreSQL", False, f"{type(erro).__name__}: {erro}")

,tabela,linhas
0,clientes,2000
1,eventos_web,50000
2,fornecedores_legado,50
3,itens_pedido,29925
4,pedidos,10000
5,produtos_catalogo,500
6,transacoes_financeiras,9986


[OK] PostgreSQL: versão 17.6, UTF8, wal_level=logical, 7 tabelas, 7 na publicação, cdc_reader com replicação=True e superuser=False


## 3. MinIO (object storage S3)

Lista os buckets e faz um teste de ida e volta: grava um Parquet pequeno (com acentos) no bucket **bronze**, lê de volta e apaga.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
from pyarrow import fs

s3 = fs.S3FileSystem(
    access_key=env("MINIO_ROOT_USER"),
    secret_key=env("MINIO_ROOT_PASSWORD"),
    endpoint_override=env("S3_ENDPOINT_URL"),
)
pasta_teste = f"{env('MINIO_BUCKET_BRONZE')}/_teste_servicos_{uuid.uuid4().hex[:8]}"

try:
    buckets = sorted(info.path for info in s3.get_file_info(fs.FileSelector("", recursive=False)))
    caminho = f"{pasta_teste}/amostra.parquet"

    tabela = pa.table({"id": [1, 2, 3], "cidade": ["São Paulo", "Maceió", "Goiânia"]})
    pq.write_table(tabela, caminho, filesystem=s3)
    lido = pq.read_table(caminho, filesystem=s3)

    ok = {env("MINIO_BUCKET_BRONZE"), env("MINIO_BUCKET_SILVER")} <= set(buckets) and lido.equals(tabela)
    registrar("MinIO", ok, f"buckets={buckets}, Parquet gravado e lido ({lido.num_rows} linhas, acentos preservados)")
except Exception as erro:
    registrar("MinIO", False, f"{type(erro).__name__}: {erro}")
finally:
    def apagar_pasta_teste():
        if s3.get_file_info(pasta_teste).type != fs.FileType.NotFound:
            s3.delete_dir(pasta_teste)

    limpar("arquivos de teste no MinIO", apagar_pasta_teste)

## 4. Kafka

Lista os tópicos existentes, cria um tópico temporário, envia uma mensagem, lê de volta e apaga o tópico.

In [ ]:
from confluent_kafka import OFFSET_BEGINNING, Consumer, Producer, TopicPartition
from confluent_kafka.admin import AdminClient, NewTopic

bootstrap = env("KAFKA_BOOTSTRAP_SERVERS")
admin = AdminClient({"bootstrap.servers": bootstrap})
topico = f"_teste_servicos_{uuid.uuid4().hex[:8]}"

try:
    topicos = sorted(t for t in admin.list_topics(timeout=10).topics if not t.startswith("__"))
    admin.create_topics([NewTopic(topico, num_partitions=1, replication_factor=1)])[topico].result(timeout=15)

    enviada = "olá, Kafka!"
    produtor = Producer({"bootstrap.servers": bootstrap})
    produtor.produce(topico, value=enviada.encode("utf-8"))
    produtor.flush(10)

    consumidor = Consumer({"bootstrap.servers": bootstrap, "group.id": topico, "enable.auto.commit": False})
    consumidor.assign([TopicPartition(topico, 0, OFFSET_BEGINNING)])  # lê a partição 0 desde o início
    mensagem, limite = None, time.time() + 20
    while mensagem is None and time.time() < limite:
        mensagem = consumidor.poll(1.0)
    consumidor.close()

    recebida = mensagem.value().decode("utf-8") if mensagem is not None and mensagem.error() is None else None
    registrar("Kafka", recebida == enviada, f"tópicos={topicos}, mensagem enviada e recebida: {recebida!r}")
except Exception as erro:
    registrar("Kafka", False, f"{type(erro).__name__}: {erro}")
finally:
    def apagar_topico_teste():
        if topico in admin.list_topics(timeout=10).topics:
            admin.delete_topics([topico])[topico].result(timeout=15)

    limpar("tópico de teste no Kafka", apagar_topico_teste)

## 5. Kafka Connect (Debezium)

Consulta a API REST do Kafka Connect: versão, se o conector **Debezium para PostgreSQL** está instalado e quais conectores já estão registrados (no início do laboratório, nenhum).

In [ ]:
def get_json(caminho):
    with urllib.request.urlopen(f"{env('KAFKA_CONNECT_URL')}{caminho}", timeout=10) as resposta:
        return json.load(resposta)


try:
    info = get_json("/")
    plugins = [p["class"] for p in get_json("/connector-plugins")]
    conectores = get_json("/connectors")
    tem_postgres = "io.debezium.connector.postgresql.PostgresConnector" in plugins
    registrar("Kafka Connect", tem_postgres,
              f"versão {info.get('version')}, conector Postgres disponível={tem_postgres}, "
              f"conectores registrados={conectores}")
except Exception as erro:
    registrar("Kafka Connect", False, f"{type(erro).__name__}: {erro}")

## 6. Iceberg (PyIceberg)

Cria uma tabela Iceberg temporária com os dados no bucket **silver** do MinIO, grava uma linha, lê de volta e apaga tudo.

O catálogo usado aqui é um SQLite **em memória**, só para o teste; o catálogo de verdade (cenário 4) fica em `data/iceberg/`.

In [ ]:
from pyiceberg.catalog.sql import SqlCatalog

namespace = f"teste_{uuid.uuid4().hex[:8]}"
nome_tabela = f"{namespace}.clientes"
warehouse = f"{env('MINIO_BUCKET_SILVER')}/_teste_servicos_iceberg_{uuid.uuid4().hex[:8]}"

catalogo = SqlCatalog(
    "teste",
    uri="sqlite:///:memory:",
    warehouse=f"s3://{warehouse}",
    **{
        "s3.endpoint": env("S3_ENDPOINT_URL"),
        "s3.access-key-id": env("MINIO_ROOT_USER"),
        "s3.secret-access-key": env("MINIO_ROOT_PASSWORD"),
        "s3.path-style-access": "true",
        "s3.region": "us-east-1",
    },
)

try:
    dados = pa.table({"id": pa.array([1], pa.int64()), "nome": ["Conceição"]})
    catalogo.create_namespace(namespace)
    tabela_iceberg = catalogo.create_table(nome_tabela, schema=dados.schema)
    tabela_iceberg.append(dados)
    lido = tabela_iceberg.scan().to_arrow()
    snapshots = len(tabela_iceberg.metadata.snapshots)
    registrar("Iceberg", lido.equals(dados),
              f"tabela criada, {lido.num_rows} linha gravada e lida, {snapshots} snapshot")
except Exception as erro:
    registrar("Iceberg", False, f"{type(erro).__name__}: {erro}")
finally:
    def apagar_tabela_teste():
        if catalogo.table_exists(nome_tabela):
            catalogo.drop_table(nome_tabela)
        if (namespace,) in catalogo.list_namespaces():
            catalogo.drop_namespace(namespace)

    def apagar_arquivos_iceberg():
        if s3.get_file_info(warehouse).type != fs.FileType.NotFound:
            s3.delete_dir(warehouse)

    limpar("tabela Iceberg de teste", apagar_tabela_teste)
    limpar("arquivos Iceberg no MinIO", apagar_arquivos_iceberg)

## 7. Resumo

In [ ]:
resumo = pd.DataFrame(resultados)
display(resumo)

falhas = resumo[resumo["status"] != "OK"]
if falhas.empty:
    print("Todos os serviços estão funcionando.")
else:
    raise RuntimeError(f"{len(falhas)} serviço(s) com falha: {', '.join(falhas['serviço'])}. Veja o detalhe acima.")